<a href="https://colab.research.google.com/github/abdulhadi2005ag-cmd/flyrank-ml-internship-hadii/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulhadi2005ag-cmd/flyrank-ml-internship-hadii/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content item, tracked by its client id and content id together, for one month. I'm using March 2026 data from fact_content_daily_performance, joined with dim_content to get word count and content type. I split the month into two halves: March 1–15 for features, March 16–31 for the label.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip install -q duckdb huggingface_hub

import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

DAILY = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')"
CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"
MARCH = f"SELECT * FROM {DAILY} WHERE month = '2026-03'"

print("Connected. Token loaded:", HF_TOKEN is not None)


Connected. Token loaded: True


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature: gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions, word_count, content_type, content_updated_date — all known before March 16, so safe to use.
Label/proxy: whether a page's impressions dropped from the first half of March to the second half. This is what I'm predicting, never a feature.
Context: client_hash_id, content_hash_id — used only to join and group, never fed to the model.
Excluded: the AI columns (sessions_ai, ai_chatgpt, ai_perplexity, ai_gemini, ai_copilot, ai_claude, ai_meta, ai_other) — too sparse across the whole warehouse (~30k rows out of 79 million) to be a useful signal here.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
schema = con.sql(f"DESCRIBE SELECT * FROM {DAILY} LIMIT 1").df()
print(schema.to_string())
schema2 = con.sql(f"DESCRIBE SELECT * FROM {CONTENT} LIMIT 1").df()
print(schema2.to_string())


                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three checks: the grain (no duplicate rows), the row count and date span of my March slice, and how many rows have usable data using IS TRUE filters.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM ({MARCH})
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Rows breaking the grain (should be empty):")
print(grain_check)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows breaking the grain (should be empty):
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []


In [ ]:
counts_check = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT content_hash_id) AS n_content_items,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM ({MARCH})
""").df()
print(counts_check)

    n_rows  n_content_items   min_date   max_date
0  9841378           331437 2026-03-01 2026-03-31


In [ ]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE AND ga4_data_available IS TRUE) AS both_available_rows
    FROM ({MARCH})
""").df()
print(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  gsc_available_rows  ga4_available_rows  both_available_rows
0     9841378             3611061              413966               364347


In [ ]:
feature_frame = con.sql(f"""
WITH h1 AS (
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions_h1,
           AVG(gsc_avg_position) AS avg_position_h1,
           SUM(ga4_sessions) AS sessions_h1
    FROM ({MARCH})
    WHERE report_date <= DATE '2026-03-15'
    GROUP BY client_hash_id, content_hash_id
),
h2 AS (
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions_h2
    FROM ({MARCH})
    WHERE report_date > DATE '2026-03-15'
    GROUP BY client_hash_id, content_hash_id
)
SELECT h1.*, h2.impressions_h2
FROM h1
JOIN h2 USING (client_hash_id, content_hash_id)
WHERE h1.impressions_h1 >= 50
""").df()

content_attrs = con.sql(f"""
    SELECT client_hash_id, content_hash_id, word_count, content_type, content_updated_date
    FROM {CONTENT}
""").df()

import pandas as pd
features = feature_frame.merge(content_attrs, on=["client_hash_id", "content_hash_id"], how="left")

features["days_since_update"] = (pd.Timestamp("2026-03-15") - pd.to_datetime(features["content_updated_date"])).dt.days
features["has_ga4_data"] = features["sessions_h1"].notna()

print(features.shape)
features[["impressions_h1", "avg_position_h1", "sessions_h1", "word_count", "days_since_update", "has_ga4_data"]].describe(include="all")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(92548, 11)


,impressions_h1,avg_position_h1,sessions_h1,word_count,days_since_update,has_ga4_data
count,92548.000000,92548.000000,49273.00000,65005.0,92548.000000,92548
unique,NaN,NaN,NaN,<NA>,NaN,2
top,NaN,NaN,NaN,<NA>,NaN,True
freq,NaN,NaN,NaN,<NA>,NaN,49273
mean,1368.890478,13.770436,10.91539,2980.834536,-66.752572,NaN
std,3323.195417,14.190372,30.53876,1129.838547,45.475061,NaN
min,50.000000,0.000000,0.00000,0.0,-113.000000,NaN
25%,143.000000,4.564959,0.00000,2475.0,-103.000000,NaN
50%,406.000000,8.065899,2.00000,2767.0,-78.000000,NaN
75%,1232.000000,18.041168,9.00000,3227.0,-64.000000,NaN


impressions_h1: known before March 16, since it's summed only from March 1–15.
avg_position_h1: known before March 16, same window.
has_ga4_data: known before March 16, flags whether GA4 tracking exists for this page.
word_count: known before the decision point — content metadata doesn't change with search performance.
days_since_update: known before the decision point — computed from content_updated_date, which is always in the past relative to March 15.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Build the label: did impressions drop from h1 to h2?
features["is_declining"] = (features["impressions_h2"] < features["impressions_h1"]).astype(int)

honest_cols = ["impressions_h1", "avg_position_h1", "word_count", "days_since_update", "has_ga4_data"]
X = features[honest_cols].fillna({"word_count": 0, "days_since_update": 0})
y = features["is_declining"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)
model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
honest_score = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
print("Honest AUC (5 real features):", honest_score)

# THE TRAP: sneak in a column computed straight from the label
features["leaky_impressions_h2"] = features["impressions_h2"]  # <-- this literally IS half the label
leaky_cols = honest_cols + ["leaky_impressions_h2"]
X_leaky = features[leaky_cols].fillna({"word_count": 0, "days_since_update": 0, "leaky_impressions_h2": 0})

X_train2, X_test2, y_train2, y_test2 = train_test_split(X_leaky, y, test_size=0.3, random_state=1)
model2 = LogisticRegression(max_iter=1000).fit(X_train2, y_train2)
leaky_score = roc_auc_score(y_test2, model2.predict_proba(X_test2)[:, 1])
print("Leaky AUC (with impressions_h2 sneaked in):", leaky_score)

print("\nScore jump:", round(leaky_score - honest_score, 3), "- this is the leakage trap. Deleting the leaky column now.")
del features["leaky_impressions_h2"]

Honest AUC (5 real features): 0.573799898862155
Leaky AUC (with impressions_h2 sneaked in): 1.0

Score jump: 0.426 - this is the leakage trap. Deleting the leaky column now.


The trap: I added leaky_impressions_h2 — literally the raw second-half impressions the label is computed from. AUC jumped from 0.57 (honest) to 1.0 (leaky) because the model could just read the label off that column instead of learning any real pattern. I deleted it and kept the honest 0.57 score. This shows how easy it is to accidentally include a feature that's really just a disguised version of the label.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice can't tell me about seasonality (only one month), can't distinguish real content decline from a client's tracking just starting or stopping (unbalanced panel — GSC/GA4 history depth differs per client), and can't reliably use GA4 sessions for most pages since only ~4% of March rows have GA4 data available. The 5-feature model's honest AUC (0.57) shows these signals alone aren't strong — a real model would need more history, more clients' worth of data, and probably query-level features from fact_content_query_90d to do meaningfully better.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
CLIENTS = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')"

history_check = con.sql(f"""
    SELECT client_hash_id, gsc_data_start, ga4_data_start
    FROM {CLIENTS}
    ORDER BY gsc_data_start
    LIMIT 10
""").df()
print(history_check)

# how much does history length vary across clients?
spread_check = con.sql(f"""
    SELECT MIN(gsc_data_start) AS earliest_gsc_start,
           MAX(gsc_data_start) AS latest_gsc_start,
           COUNT(*) AS n_clients
    FROM {CLIENTS}
""").df()
print(spread_check)


            client_hash_id gsc_data_start ga4_data_start
0  client_9958f0a7ae1df715     2025-01-27     2025-10-29
1  client_ff644d8251367cbb     2025-01-27     2025-10-29
2  client_73cda7b4e4f265ea     2025-02-11     2026-03-24
3  client_fef1a8f436438636     2025-03-11     2026-03-06
4  client_62f4a7e64f5e0096     2025-06-07            NaT
5  client_b10cb2997d0c7c86     2025-06-18     2025-11-15
6  client_65de48885f4ef01b     2025-06-21     2026-02-19
7  client_c182d11e4862a37d     2025-06-21     2026-02-20
8  client_3197e6291363b4db     2025-06-29     2025-11-09
9  client_625b6439094e23e4     2025-07-01     2026-02-19
  earliest_gsc_start latest_gsc_start  n_clients
0         2025-01-27       2026-06-02        104


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.